<a href="https://colab.research.google.com/github/pratikshyasarangi/bshape/blob/main/B-Shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import itertools
import math

# Define the 6 specific risk factors in your model
FACTORS = ['Exp', 'Rtg', 'LGD_seg', 'Util', 'Flag', 'Scen']

def calculate_combinatoric_weight(subset_size, total_factors):
    """
    Calculates the exact Shapley combinatoric probability weight.
    Formula: (|S|! * (|N| - |S| - 1)!) / |N|!
    """
    return (math.factorial(subset_size) * math.factorial(total_factors - subset_size - 1)) / math.factorial(total_factors)

def mock_expected_loss(state):
    """
    A mock expected loss engine representing your nested model architecture.
    Master Equation: Loss = Exp * PD(Rtg, Exp, Scen) * LGD(LGD_seg, Scen) * EAD(Util, Flag, Exp, Scen)
    """
    # Extract variables from the state dictionary
    exp = state['Exp']
    rtg = state['Rtg']
    lgd_seg = state['LGD_seg']
    util = state['Util']
    flag = state['Flag']
    scen = state['Scen']

    # 1. EAD Calculation
    # EAD = (Used + LEQ * Unused) / Exposure
    unused = max(exp - util, 0)

    # Nested LEQ scalar based on Flag and Scenario
    if flag == 'Revolver':
        leq = 0.5 if scen == 'Base' else 0.75
    elif flag == 'SBLC':
        leq = 0.2 if scen == 'Base' else 0.4
    else:
        leq = 0.0

    if flag in ['Revolver', 'SBLC']:
        ead_pct = (util + (leq * unused)) / exp if exp > 0 else 0
    else:
        ead_pct = 1.0

    # 2. PD Calculation
    # PD = matrix(Rtg) * stress(Scen) * size_factor(Rtg, Exp)
    base_pd = 0.05 if rtg == 'Substandard' else 0.01
    pd_stress = 1.5 if scen == 'Credit Crisis' else 1.0

    # Nested PD Size Factor dependency
    size_factor = 1.2 if exp > 1000000 else 1.0

    pd = base_pd * pd_stress * size_factor

    # 3. LGD Calculation
    # LGD = base_lgd(LGD_seg) * lgd_stress(Scen)
    base_lgd = 0.45 if lgd_seg == 'Unsecured' else 0.20
    lgd_stress = 1.2 if scen == 'Credit Crisis' else 1.0
    lgd = min(base_lgd * lgd_stress, 1.0) # Cap severity at 100%

    # 4. Master Loss Function Execution
    return exp * pd * lgd * ead_pct


def calculate_b_shapley_attribution(state_m1, state_m2):
    """
    Calculates the completely order-independent B-Shapley attribution for the
    change in expected loss from Month 1 to Month 2.
    """
    attributions = {factor: 0.0 for factor in FACTORS}
    total_factors = len(FACTORS)

    # Iterate over each factor 'k' to calculate its isolated Shapley value
    for k in FACTORS:

        # Axiom of the Dummy Player: If the factor hasn't changed, its attribution is exactly 0
        if state_m1[k] == state_m2[k]:
            continue

        other_factors = [f for f in FACTORS if f != k]

        # Generate all 2^(N-1) possible binary subsets of the REMAINING factors
        for subset_size in range(len(other_factors) + 1):
            for subset in itertools.combinations(other_factors, subset_size):

                # 1. Create a hybrid state WITHOUT updating factor k (k remains at Month 1)
                state_without_k = {}
                for f in FACTORS:
                    if f in subset:
                        state_without_k[f] = state_m2[f] # Updated to Month 2
                    else:
                        state_without_k[f] = state_m1[f] # Kept at Month 1

                # 2. Create the exact same hybrid state, but WITH updating factor k to Month 2
                state_with_k = state_without_k.copy()
                state_with_k[k] = state_m2[k]

                # Evaluate your black-box loss function for both states
                loss_without_k = mock_expected_loss(state_without_k)
                loss_with_k = mock_expected_loss(state_with_k)

                # Calculate marginal impact
                marginal_impact = loss_with_k - loss_without_k

                # Apply combinatoric probability weight
                weight = calculate_combinatoric_weight(len(subset), total_factors)

                # Add to the running total for factor k
                attributions[k] += weight * marginal_impact

    return attributions


# ==========================================
# Example Execution and Verification
# ==========================================
if __name__ == "__main__":

    # Define Month 1 (Base State)
    month_1_data = {
        'Exp': 1000000,
        'Rtg': 'Pass',
        'LGD_seg': 'Secured',
        'Util': 500000,
        'Flag': 'Revolver',
        'Scen': 'Base'
    }

    # Define Month 2 (Target State)
    month_2_data = {
        'Exp': 1500000,           # Exposure increased (Triggers PD size factor update + EAD denominator shift)
        'Rtg': 'Substandard',     # Credit Quality deteriorated
        'LGD_seg': 'Secured',     # Unchanged (Dummy Player)
        'Util': 1200000,          # Utilization increased
        'Flag': 'Revolver',       # Unchanged (Dummy Player)
        'Scen': 'Credit Crisis'   # Scenario shifted, altering PD, LGD, and LEQ stress scalars
    }

    # Calculate Total Loss using the core engine
    loss_m1 = mock_expected_loss(month_1_data)
    loss_m2 = mock_expected_loss(month_2_data)
    total_delta = loss_m2 - loss_m1

    # Calculate B-Shapley Attributions
    attributions = calculate_b_shapley_attribution(month_1_data, month_2_data)

    # Print Results
    print(f"Month 1 Expected Loss: ${loss_m1:,.2f}")
    print(f"Month 2 Expected Loss: ${loss_m2:,.2f}")
    print(f"Total Loss Delta:      ${total_delta:,.2f}\n")

    print("--- Order-Independent Factor Attribution ---")
    sum_attributions = 0
    for factor, impact in attributions.items():
        print(f"{factor:10} Impact: ${impact:12,.2f}")
        sum_attributions += impact

    print("-" * 37)
    print(f"Sum of Attributions:   ${sum_attributions:,.2f}")

    # Verification Step: Guaranteeing Additive Efficiency
    unexplained = total_delta - sum_attributions
    print(f"Unexplained Residual:  ${abs(unexplained):,.2f}")

Month 1 Expected Loss: $1,500.00
Month 2 Expected Loss: $30,780.00
Total Loss Delta:      $29,280.00

--- Order-Independent Factor Attribution ---
Exp        Impact: $    4,593.00
Rtg        Impact: $   14,626.00
LGD_seg    Impact: $        0.00
Util       Impact: $    2,613.00
Flag       Impact: $        0.00
Scen       Impact: $    7,448.00
-------------------------------------
Sum of Attributions:   $29,280.00
Unexplained Residual:  $0.00
